# Allegro Reproduction Results Analysis

This notebook analyzes the results from reproducing the Allegro Nature Communications paper.

**Paper:** Learning local equivariant representations for large-scale atomistic dynamics  
**Authors:** Musaelian et al.  
**Journal:** Nature Communications (2023)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. QM9 Results

### 1.1 Load QM9 Results

In [ ]:
# Define paper results (in meV)
paper_results_qm9 = {
    'U0': 4.0,
    'U': 4.0,
    'H': 4.0,
    'G': 5.0
}

# Load your results here
# TODO: Update these paths to your actual results
our_results_qm9 = {
    'U0': None,  # Update with your result
    'U': None,
    'H': None,
    'G': None
}

# Example of loading from a results file:
# with open('../results/qm9/U0/metrics.json', 'r') as f:
#     metrics = json.load(f)
#     our_results_qm9['U0'] = metrics['test_mae'] * 1000  # Convert to meV if needed

### 1.2 Compare QM9 Results

In [ ]:
# Create comparison DataFrame
qm9_comparison = pd.DataFrame({
    'Target': list(paper_results_qm9.keys()),
    'Paper (meV)': list(paper_results_qm9.values()),
    'Ours (meV)': list(our_results_qm9.values())
})

# Calculate differences
qm9_comparison['Difference (meV)'] = qm9_comparison['Ours (meV)'] - qm9_comparison['Paper (meV)']
qm9_comparison['Relative Error (%)'] = (qm9_comparison['Difference (meV)'] / qm9_comparison['Paper (meV)']) * 100

print("QM9 Results Comparison:")
print(qm9_comparison.to_string(index=False))

### 1.3 Visualize QM9 Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot comparison
x = np.arange(len(qm9_comparison))
width = 0.35

axes[0].bar(x - width/2, qm9_comparison['Paper (meV)'], width, label='Paper', alpha=0.8)
axes[0].bar(x + width/2, qm9_comparison['Ours (meV)'], width, label='Our Reproduction', alpha=0.8)
axes[0].set_xlabel('Target')
axes[0].set_ylabel('MAE (meV)')
axes[0].set_title('QM9 Results: Paper vs Reproduction')
axes[0].set_xticks(x)
axes[0].set_xticklabels(qm9_comparison['Target'])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Difference plot
axes[1].bar(qm9_comparison['Target'], qm9_comparison['Difference (meV)'])
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Target')
axes[1].set_ylabel('Difference (meV)')
axes[1].set_title('Difference: Our Results - Paper Results')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/qm9_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. revMD17 Results

### 2.1 Load revMD17 Results

In [ ]:
# Define paper results
paper_results_md17 = {
    'Energy MAE (meV)': 3.84,
    'Energy Std (meV)': 0.08,
    'Force MAE (meV/Å)': 12.98,
    'Force Std (meV/Å)': 0.17
}

# Load your results here
# TODO: Update with your actual results
our_results_md17 = {
    'aspirin': {'energy_mae': None, 'force_mae': None},
    'benzene': {'energy_mae': None, 'force_mae': None},
    'ethanol': {'energy_mae': None, 'force_mae': None},
    'malonaldehyde': {'energy_mae': None, 'force_mae': None},
    'naphthalene': {'energy_mae': None, 'force_mae': None},
    'salicylic_acid': {'energy_mae': None, 'force_mae': None},
    'toluene': {'energy_mae': None, 'force_mae': None},
    'uracil': {'energy_mae': None, 'force_mae': None}
}

### 2.2 Analyze revMD17 Results

In [ ]:
# Calculate averages
molecules = list(our_results_md17.keys())
energy_maes = [our_results_md17[mol]['energy_mae'] for mol in molecules if our_results_md17[mol]['energy_mae'] is not None]
force_maes = [our_results_md17[mol]['force_mae'] for mol in molecules if our_results_md17[mol]['force_mae'] is not None]

if energy_maes:
    our_energy_mean = np.mean(energy_maes)
    our_energy_std = np.std(energy_maes)
    our_force_mean = np.mean(force_maes)
    our_force_std = np.std(force_maes)
    
    print(f"Our Results (averaged over {len(energy_maes)} molecules):")
    print(f"  Energy MAE: {our_energy_mean:.2f} ± {our_energy_std:.2f} meV")
    print(f"  Force MAE: {our_force_mean:.2f} ± {our_force_std:.2f} meV/Å")
    print()
    print(f"Paper Results:")
    print(f"  Energy MAE: {paper_results_md17['Energy MAE (meV)']:.2f} ± {paper_results_md17['Energy Std (meV)']:.2f} meV")
    print(f"  Force MAE: {paper_results_md17['Force MAE (meV/Å)']:.2f} ± {paper_results_md17['Force Std (meV/Å)']:.2f} meV/Å")

## 3. Summary and Conclusions

In [ ]:
print("="*60)
print("REPRODUCTION SUMMARY")
print("="*60)
print()
print("QM9 Benchmark:")
print("-"*60)
for _, row in qm9_comparison.iterrows():
    print(f"  {row['Target']:5s}: Paper = {row['Paper (meV)']:5.1f} meV, Ours = {row['Ours (meV)']:5.1f} meV, Diff = {row['Difference (meV)']:+5.2f} meV ({row['Relative Error (%)']:+5.1f}%)")
print()
print("revMD17 Benchmark:")
print("-"*60)
if energy_maes:
    print(f"  Energy MAE: Paper = {paper_results_md17['Energy MAE (meV)']:.2f} meV, Ours = {our_energy_mean:.2f} meV")
    print(f"  Force MAE:  Paper = {paper_results_md17['Force MAE (meV/Å)']:.2f} meV/Å, Ours = {our_force_mean:.2f} meV/Å")
print()
print("="*60)
print("Status: [To be filled after completing all experiments]")
print("="*60)

## 4. Export Results

Save results to CSV for documentation.

In [ ]:
# Save QM9 comparison
qm9_comparison.to_csv('../results/qm9_comparison.csv', index=False)
print("QM9 comparison saved to: ../results/qm9_comparison.csv")

# Save MD17 results
if energy_maes:
    md17_df = pd.DataFrame({
        'Molecule': molecules[:len(energy_maes)],
        'Energy MAE (meV)': energy_maes,
        'Force MAE (meV/Å)': force_maes
    })
    md17_df.to_csv('../results/revmd17_results.csv', index=False)
    print("revMD17 results saved to: ../results/revmd17_results.csv")